## Figure 9d — February days-21–40 EP100 precursor

Input: canonical `relationships/figure09d.csv` produced by analysis notebook 05. It contains exactly 30 February-initialized members, the mean upward 100-hPa EP flux over forecast days 21–40 (21 February–12 March), the centered-five-day March–April ozone minimum, stored Pearson/OLS statistics, and the no-W year-0008 reference point. The reference point is not part of the 30-member fit. No Spearman statistic is used.

Output: `figure09d.png` and `figure09d.pdf`, using the shared publication-scale style. Member identifiers are omitted; only the no-W reference is labelled `ref 0008`.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def discover_repository_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "analysis").is_dir() and (candidate / "figures").is_dir():
            return candidate.resolve()
    raise RuntimeError("Run this notebook from the Paper 1 code checkout or figures directory")


REPOSITORY_ROOT = discover_repository_root()
DERIVED_ROOT = Path(
    os.environ.get("PAPER1_DERIVED_ROOT", str(REPOSITORY_ROOT / "work"))
).expanduser().resolve()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
PRODUCT_VERSION = "Paper1_828_repro_v1"
if DERIVED_ROOT == Path(DERIVED_ROOT.anchor):
    raise PermissionError("PAPER1_DERIVED_ROOT cannot be a filesystem root")
if OUTPUT_DIR != DERIVED_ROOT and DERIVED_ROOT not in OUTPUT_DIR.parents:
    raise PermissionError("PAPER1_FIGURE_ROOT must be below PAPER1_DERIVED_ROOT")


path = DERIVED_ROOT / "relationships" / "figure09d.csv"
if not path.is_file():
    raise FileNotFoundError(f"Missing canonical Figure 9d product: {path}")
table = pd.read_csv(path)
required = {
    "product_version", "case", "member", "x", "y", "r", "p", "n",
    "slope", "intercept", "ref_x", "ref_y", "window_start_doy",
    "window_end_doy", "window_days", "forecast_day_start", "forecast_day_end",
    "epflux_method", "minimum_method",
}
missing = sorted(required.difference(table.columns))
if missing:
    raise ValueError(f"{path} is missing columns {missing}")
if set(table["product_version"].astype(str)) != {PRODUCT_VERSION}:
    raise ValueError(f"{path}: unexpected product_version")
if len(table) != 30 or table["member"].astype(str).nunique() != 30:
    raise ValueError("Figure 9d requires exactly 30 unique February members")
if set(table["case"].astype(str)) != {"0008-02"}:
    raise ValueError("Figure 9d must use the February-initialized case 0008-02")
expected_constants = {
    "window_start_doy": 52, "window_end_doy": 71, "window_days": 20,
    "forecast_day_start": 21, "forecast_day_end": 40, "n": 30,
}
for column, expected in expected_constants.items():
    if set(pd.to_numeric(table[column], errors="raise").astype(int)) != {expected}:
        raise ValueError(f"{path}: {column} must equal {expected}")
if not table["epflux_method"].astype(str).str.contains("w=None", regex=False).all():
    raise ValueError("Figure 9d requires the stored no-W EP-flux diagnostic")
if not table["minimum_method"].astype(str).str.contains("centered5", case=False).all():
    raise ValueError("Figure 9d requires the centered-5-day spring O3 minimum")
if any("spearman" in column.lower() for column in table.columns):
    raise ValueError("Figure 9d permits Pearson statistics only")

x = pd.to_numeric(table["x"], errors="raise").to_numpy(float)
y = pd.to_numeric(table["y"], errors="raise").to_numpy(float)
r = float(table["r"].iloc[0])
p = float(table["p"].iloc[0])
slope = float(table["slope"].iloc[0])
intercept = float(table["intercept"].iloc[0])
ref_x = float(table["ref_x"].iloc[0])
ref_y = float(table["ref_y"].iloc[0])
if not all(np.isfinite(value) for value in (r, p, slope, intercept, ref_x, ref_y)):
    raise ValueError("Figure 9d stored statistics/reference point must be finite")

figure, axis = plt.subplots(figsize=(10.4, 7.0))
axis.scatter(x, y, s=38, color="#2f6fb0", edgecolor="white", linewidth=0.5, zorder=3)
fit_x = np.linspace(float(np.nanmin(x)), float(np.nanmax(x)), 200)
axis.plot(fit_x, intercept + slope * fit_x, color="#c43c39", linewidth=1.5, zorder=2)
axis.scatter(
    [ref_x], [ref_y], marker="*", s=150, color="#f2c14e",
    edgecolor="black", linewidth=0.7, label="WACCM year 0008 (no-W reference)", zorder=5,
)
axis.annotate(
    "ref 0008", (ref_x, ref_y), xytext=(3, 3),
    textcoords="offset points", fontsize=6.5, color="0.22",
)
axis.text(
    0.03, 0.97, f"Pearson r = {r:.2f}\np = {p:.3f}\nn = 30",
    transform=axis.transAxes, ha="left", va="top",
    bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "alpha": 0.85, "edgecolor": "0.65"},
)
axis.set_xlabel(r"Mean upward EP flux at 100 hPa, 21 Feb–12 Mar (W m$^{-2}$)")
axis.set_ylabel(r"Centered-5-day spring O$_3$ minimum (DU)")
axis.set_title("February hindcast: days-21–40 EP100 versus spring ozone minimum")
axis.grid(color="0.90", linewidth=0.6)
axis.legend(loc="best", frameon=True)

import sys as _sys
style_directory = str(REPOSITORY_ROOT / "figures")
if style_directory not in _sys.path:
    _sys.path.insert(0, style_directory)
from paper_style import apply_paper_style
apply_paper_style(figure, "figure09d")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for suffix, kwargs in (("png", {"dpi": 300}), ("pdf", {})):
    destination = OUTPUT_DIR / f"figure09d.{suffix}"
    temporary = OUTPUT_DIR / f".figure09d.{os.getpid()}.{suffix}.tmp"
    try:
        figure.savefig(temporary, format=suffix, bbox_inches="tight", facecolor="white", **kwargs)
        if temporary.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {temporary}")
        os.replace(temporary, destination)
    finally:
        if temporary.exists():
            temporary.unlink()
    print(f"saved {destination}")
plt.close(figure)
